# Thuc hanh 2 - Phan tich Log File

Notebook nay chay duoc ngay trong Jupyter.

- Doc file `server.log` trong thu muc notebook
- Day du lieu len Spark cluster bang `parallelize`
- Loc dong `ERROR` va thong ke theo loai loi


In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName('LogAnalysisNotebook')
    .master('spark://master:7077')
    .config('spark.pyspark.python', '/opt/conda/bin/python')
    .getOrCreate()
)

sc = spark.sparkContext
print('Spark master =', sc.master)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/03 12:50:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/04/03 12:50:42 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/03 12:50:42 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


Spark master = spark://master:7077


In [2]:
input_file = '/opt/workspace/notebooks/server.log'

with open(input_file, encoding='utf-8') as f:
    log_lines = [line.strip() for line in f if line.strip()]

print('So dong log doc duoc:', len(log_lines))
for line in log_lines:
    print(line)


So dong log doc duoc: 5
2024-01-15 10:23:01 ERROR DatabaseConnection timeout
2024-01-15 10:23:05 INFO User login successful
2024-01-15 10:23:10 ERROR FileNotFound /data/report.csv
2024-01-15 10:23:15 WARN Memory usage 85%
2024-01-15 10:23:20 ERROR DatabaseConnection refused


In [3]:
# parallelize day du lieu tu driver len cluster de worker xu ly.
logs = sc.parallelize(log_lines, 4)
errors = logs.filter(lambda line: 'ERROR' in line)
total_errors = errors.count()

error_types = errors.map(lambda line: (line.split()[3], 1))
error_counts = error_types.reduceByKey(lambda a, b: a + b)
result = error_counts.sortBy(lambda item: (-item[1], item[0])).collect()

print(f'Tong so loi: {total_errors}')
print('LOG ANALYSIS RESULT')
for error_type, count in result:
    print(f'{error_type}\t{count}')


26/04/03 12:50:58 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources


26/04/03 12:51:13 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources


26/04/03 12:51:28 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources


26/04/03 12:51:43 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources


26/04/03 12:51:58 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources


26/04/03 12:52:13 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources


26/04/03 12:52:28 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources


Tong so loi: 3
LOG ANALYSIS RESULT
DatabaseConnection	2
FileNotFound	1


In [4]:
# Neu muon chay lai tu dau, restart kernel roi chay lai notebook.
# spark.stop()
